# THIS NEEDS TO COMPLETELY BE FIXED
# 5.1 Class area summary (km² and % cover)

Counts the pixels of each typology class in every Time Interval raster (`TimeInterval*.tif`)
and in the newest Time Series raster, converts them to km² and % cover, and writes one CSV
with all rasters side by side.

- Projection: South America Albers Equal Area Conic (ESRI:102033)
- Cell size: 31.887054 m, so one pixel = **1016.79 m²** (not 900 m²)
- Area = pixel count × 1016.79 m²; % cover = class area / total classified area of that raster

**Packages** (all already in `environment.yml`): `numpy`, `pandas`, `rasterio`

In [ ]:
### Packages ###
import time
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window

In [161]:
### Parameters ###
TYP = Path(r"./data/5_Typology")                                   # folder with the typology rasters
TS_PATH = TYP / "combined_rc_timeseries" / "combined_rc.tif"       # newest Time Series typology
OUT_CSV = TYP / "class_area_km2_summary.csv"

PX_M2 = 1016.79        # m2 per pixel (31.887054 m x 31.887054 m, ESRI:102033)
ROWS = 128             # rows read at a time; lower if RAM is tight

CLASSES = {1: "Attrition", 2: "Aggregation", 3: "Shrinkage", 4: "Perforation", 5: "Deformation",
           6: "Persistent", 7: "Enlargement", 8: "Dissection", 9: "Fragmentation per se", 10: "Creation"}

In [162]:
### Find the rasters ###
RASTERS = {p.stem: p for p in sorted(TYP.glob("TimeInterval*.tif"))}
RASTERS["TimeSeries 1990-2020"] = TS_PATH

print("rasters to summarize:")
for name, p in RASTERS.items():
    print("  ", "OK     " if p.exists() else "MISSING", name, "->", p)
assert all(p.exists() for p in RASTERS.values()), "a raster is missing; check the paths above"

rasters to summarize:
   OK      TimeInterval-1990-1995 -> data/5_Typology/TimeInterval-1990-1995.tif
   OK      TimeInterval-1990-2020 -> data/5_Typology/TimeInterval-1990-2020.tif
   OK      TimeInterval-1995-2000 -> data/5_Typology/TimeInterval-1995-2000.tif
   OK      TimeInterval-2000-2005 -> data/5_Typology/TimeInterval-2000-2005.tif
   OK      TimeInterval-2005-2010 -> data/5_Typology/TimeInterval-2005-2010.tif
   OK      TimeInterval-2010-2015 -> data/5_Typology/TimeInterval-2010-2015.tif
   OK      TimeInterval-2015-2020 -> data/5_Typology/TimeInterval-2015-2020.tif
   OK      TimeSeries 1990-2020 -> data/5_Typology/combined_rc_timeseries/combined_rc.tif


In [163]:
### Functions ###
def check_raster(path):
    """Print the CRS and pixel size, and warn if the pixel area is not PX_M2."""
    with rasterio.open(path) as src:
        px = abs(src.transform.a * src.transform.e)
        print("  CRS:", src.crs, "| cell size:", src.res, "| pixel m2:", round(px, 2))
        if abs(px - PX_M2) > 0.01:
            print("  WARNING: pixel area is", round(px, 2), "m2, not", PX_M2)


def count_classes(path):
    """Pixel count per class value (NoData excluded), read in strips."""
    counts = np.zeros(256, np.int64)
    with rasterio.open(path) as src:
        for r in range(0, src.height, ROWS):
            w = Window(0, r, src.width, min(ROWS, src.height - r))
            a = src.read(1, window=w, masked=True)
            v = np.ma.getdata(a)[~np.ma.getmaskarray(a)].astype(np.int64)
            counts += np.bincount(v[(v >= 0) & (v < 256)], minlength=256)
    other = int(counts.sum() - sum(counts[k] for k in CLASSES))
    if other:
        print("  WARNING:", f"{other:,}", "pixels have values that are not classes 1-10")
    return {k: int(counts[k]) for k in CLASSES}


def class_table(counts):
    """Pixels, km2 and % cover per class, plus a TOTAL row."""
    df = pd.DataFrame({"Class": list(CLASSES.values()), "pixels": list(counts.values())})
    total = df["pixels"].sum()
    df["km2"] = df["pixels"] * PX_M2 / 1e6
    df["%cover"] = 100 * df["pixels"] / total
    df.loc[len(df)] = ["TOTAL", total, total * PX_M2 / 1e6, 100.0]
    return df

In [164]:
### Run: count every raster ###
tables = {}
for name, p in RASTERS.items():
    t0 = time.time()
    print(name)
    check_raster(p)
    tables[name] = class_table(count_classes(p))
    print(f"  done in {(time.time() - t0) / 60:.1f} min")

TimeInterval-1990-1995
  CRS: ESRI:102033 | cell size: (31.8870543761473, 31.887054376147304) | pixel m2: 1016.78
  done in 0.7 min
TimeInterval-1990-2020
  CRS: ESRI:102033 | cell size: (31.8870543761473, 31.887054376147304) | pixel m2: 1016.78
  done in 0.6 min
TimeInterval-1995-2000
  CRS: ESRI:102033 | cell size: (31.8870543761473, 31.887054376147304) | pixel m2: 1016.78
  done in 0.7 min
TimeInterval-2000-2005
  CRS: ESRI:102033 | cell size: (31.8870543761473, 31.887054376147304) | pixel m2: 1016.78
  done in 0.7 min
TimeInterval-2005-2010
  CRS: ESRI:102033 | cell size: (31.8870543761473, 31.887054376147304) | pixel m2: 1016.78
  done in 0.6 min
TimeInterval-2010-2015
  CRS: ESRI:102033 | cell size: (31.8870543761473, 31.887054376147304) | pixel m2: 1016.78
  done in 0.6 min
TimeInterval-2015-2020
  CRS: ESRI:102033 | cell size: (31.8870543761473, 31.887054376147304) | pixel m2: 1016.78
  done in 0.7 min
TimeSeries 1990-2020
  CRS: ESRI:102033 | cell size: (31.8870543761473, 31.8

In [165]:
### Combine into one table and save the CSV ###
combined = pd.DataFrame({"Class": list(CLASSES.values()) + ["TOTAL"]})
for name, t in tables.items():
    combined[f"{name} pixels"] = t["pixels"].values
    combined[f"{name} km2"] = t["km2"].round(1).values
    combined[f"{name} %cover"] = t["%cover"].round(4).values

combined.to_csv(OUT_CSV, index=False)
print("saved", OUT_CSV)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
combined

saved data/5_Typology/class_area_km2_summary.csv


,Class,TimeInterval-1990-1995 pixels,TimeInterval-1990-1995 km2,TimeInterval-1990-1995 %cover,TimeInterval-1990-2020 pixels,TimeInterval-1990-2020 km2,TimeInterval-1990-2020 %cover,TimeInterval-1995-2000 pixels,TimeInterval-1995-2000 km2,TimeInterval-1995-2000 %cover,TimeInterval-2000-2005 pixels,TimeInterval-2000-2005 km2,TimeInterval-2000-2005 %cover,TimeInterval-2005-2010 pixels,TimeInterval-2005-2010 km2,TimeInterval-2005-2010 %cover,TimeInterval-2010-2015 pixels,TimeInterval-2010-2015 km2,TimeInterval-2010-2015 %cover,TimeInterval-2015-2020 pixels,TimeInterval-2015-2020 km2,TimeInterval-2015-2020 %cover,TimeSeries 1990-2020 pixels,TimeSeries 1990-2020 km2,TimeSeries 1990-2020 %cover
0,Attrition,186363763,189492.8,18.9887,186175860,189301.8,18.2690,182101771,185159.3,18.6142,150069191,152588.9,15.3615,122823859,124886.1,12.5234,123839971,125919.2,12.5391,125343020,127447.5,12.6328,137828956,140143.1,13.7795
1,Aggregation,113715855,115625.1,11.5865,139984600,142334.9,13.7363,113882229,115794.3,11.6409,123792494,125871.0,12.6717,124207944,126293.4,12.6645,118879189,120875.2,12.0368,113307437,115209.9,11.4198,177263385,180239.6,17.7220
2,Shrinkage,115473369,117412.2,11.7656,73565852,74801.0,7.2188,113773353,115683.6,11.6297,109101396,110933.2,11.1679,100657927,102348.0,10.2633,99697376,101371.3,10.0946,103942564,105687.8,10.4760,70492349,71675.9,7.0475
3,Perforation,44796201,45548.3,4.5643,30363483,30873.3,2.9795,42794342,43512.9,4.3744,40447895,41127.0,4.1404,42303756,43014.0,4.3134,45924711,46695.8,4.6500,48725278,49543.4,4.9108,18637559,18950.5,1.8633
4,Deformation,5382203,5472.6,0.5484,2291771,2330.2,0.2249,5562551,5655.9,0.5686,5970031,6070.3,0.6111,6143871,6247.0,0.6264,6265108,6370.3,0.6344,6617143,6728.2,0.6669,36176961,36784.4,3.6168
5,Persistent,70071189,71247.7,7.1396,25909886,26344.9,2.5425,71942660,73150.6,7.3539,75388114,76653.9,7.7169,80835500,82192.7,8.2422,76455190,77738.9,7.7413,79878498,81219.7,8.0506,107997320,109810.6,10.7971
6,Enlargement,159930386,162615.6,16.2954,112341729,114227.9,11.0238,167475644,170287.6,17.1191,182242221,185302.1,18.6548,197353612,200667.2,20.1226,193642611,196893.9,19.6068,199946489,203303.6,20.1518,108100488,109915.5,10.8074
7,Dissection,141742154,144122.0,14.4422,188005937,191162.6,18.4486,135673743,137951.7,13.8684,118647222,120639.3,12.1451,108094052,109909.0,11.0215,117633484,119608.6,11.9107,121299197,123335.8,12.2253,108158904,109974.9,10.8132
8,Fragmentation per se,1834006,1864.8,0.1869,1357667,1380.5,0.1332,1852285,1883.4,0.1893,2069788,2104.5,0.2119,2211959,2249.1,0.2255,2394085,2434.3,0.2424,2435944,2476.8,0.2455,65751275,66855.2,6.5735
9,Creation,142137980,144524.5,14.4825,259085114,263435.2,25.4234,143238439,145643.4,14.6416,169189520,172030.2,17.3187,196122358,199415.3,19.9971,202895899,206302.5,20.5438,190703797,193905.7,19.2203,169839364,172691.0,16.9797
